# 01 — Dataset provenance, licensing, and immutable inputs

**Estimated time:** 30 minutes<br>
**Prerequisites:** 00 — Start here<br>
**Learner-produced evidence:** a source-contract review and verified local SHA-256 values

## Learning objectives

- Read the verified dataset identity and licensing record.
- Explain why public access is not commercial or production clearance.
- Verify immutable local inputs without printing source records.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


## Why this matters

A model result is unusable if nobody can establish where its data came from or whether that use was permitted. Provenance and license review happen before exploration because inspection, redistribution, training, and commercial use can have different permissions. This is a governance gate, not paperwork added after a model succeeds.

## Key terms in plain language

- **dataset:** a defined collection of records used for a stated purpose.
- **provenance:** the chain of custody describing the source, acquisition method, version, and transformations of data.
- **license:** the legal terms under which a rights holder permits specified uses; it is not a quality or ethics certificate.
- **permitted use:** a concrete activity—such as local study, modification, redistribution, or model training—that the relevant terms and policy allow.
- **dataset card:** documentation of a dataset's contents, intended uses, origin, limitations, license metadata, and known risks.
- **immutable raw input:** the acquired bytes preserved without hand edits so later transformations remain auditable.
- **SHA-256 fingerprint:** a deterministic digest of bytes used to detect change; it identifies content but does not prove that the content is safe, correct, or lawful.


## Mental model — how to think about this

Think like an evidence custodian receiving a sealed package. First record who supplied it, under what terms, on what date, and with what fingerprint. Then keep the package unchanged and derive working copies through repeatable transformations. Apply two independent gates: **may we use it?** and **is it fit for this purpose?** Passing one never implies the other.

### Running example

Follow that password-reset row through its chain of custody: a named Kaggle dataset revision contains a downloaded archive; the archive contains a CSV; the preserved bytes have a recorded SHA-256; repeatable code later derives a validated record. The hash answers `same bytes?`, not `safe and lawful?`.

### Questions to ask before continuing

- Can the exact bytes and source version be identified again?
- Do the verified terms cover this activity and this audience, including redistribution?
- Which statements are source facts, which are interpretations, and who approved them?
- If a term is missing or ambiguous, should the workflow stop or escalate to an owner?


## Current best practices

**Guidance reviewed:** 2026-08-01. These are reasons to inspect future tool changes, not a claim that practice stops evolving.

- **Verify the current primary source.** Save the source URL or identifier, observed date, version, and license text or authoritative reference before use.
- **Hash the acquired bytes.** A content fingerprint makes silent replacement visible and connects every derivative artifact to a specific input.
- **Never edit raw data in place.** Normalize and filter into a separate interim or processed layer with recorded code and configuration.
- **Document intended and out-of-scope uses.** A useful dataset card covers limitations, sensitive content, collection context, and known representational gaps—not just columns.
- **Fail closed on unclear rights.** Record `not verified` rather than inferring permission from public visibility or from a repository filename.

## Common mistakes and why they fail

- **Public means free for every use.** Public access and legal permission are different facts.
- **A license field settles all rights.** Hosted content can include third-party material, privacy obligations, or terms that require separate review.
- **The filename is a version.** Mutable files can keep the same name; preserve a digest and source revision.
- **License review proves responsible use.** It does not establish consent, fairness, data quality, security, or suitability for deployment.

### What kind of guidance is this?

A **specification** defines a technical contract; **tool guidance** describes current official library behavior; **risk guidance** is voluntary governance guidance; and a **course rule** is this project's deliberately conservative choice. Do not call all four a formal standard. The lesson is complete offline; these primary links are optional follow-up reading.

- **Tool guidance:** [Hugging Face Dataset Cards documentation](https://huggingface.co/docs/hub/en/datasets-cards)
- **Risk guidance:** [NIST AI RMF Generative AI Profile](https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-generative-artificial-intelligence)


## Setup — run, do not edit

Run the next cell once. It verifies the dedicated local Python kernel, finds
this sample project, and enables supported offline flags **before** model or
tracking libraries are imported. A successful cell ends with `setup: ready`.

This is one defense layer, not proof that every native library is physically
incapable of networking. The flight-preparation manifest, cached assets,
socket-denial checks, and a Wi-Fi-off rehearsal provide the other layers.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

{
    "setup": "ready",
    "kernel": "AAI Local Fine-Tuning (offline)",
    "python": str(active_python),
    "network_library_flags": "enabled",
    "note": "Continue to the lesson; this cell is setup, not an exercise.",
}

## Read the source contract before the rows

Provenance comes before modelling. The tracked card records title,
owner, current source URL, version, license, intended curriculum use,
redistribution constraints, source composition, quality findings, and
limitations. Settings contain public identifiers and hashes—not secrets.


In [ ]:
from aai_local_finetuning.settings import (
    PROJECT_ROOT,
    load_settings,
    sha256_file,
)

settings = load_settings()
card_path = PROJECT_ROOT / "dataset_cards" / "bitext-customer-support.md"
source_contract = {
    "title": settings.dataset.title,
    "owner": settings.dataset.owner,
    "source": settings.dataset.url,
    "kaggle_version": settings.dataset.version,
    "license_recorded": settings.dataset.license,
    "accessed_on": settings.dataset.accessed_on,
    "language": settings.dataset.language,
    "dataset_card": str(card_path.relative_to(PROJECT_ROOT)),
}
source_contract

## Verify bytes, not filenames

A filename can be silently replaced. SHA-256 binds later results to the
exact archive and CSV inspected for the course. Hashing is read-only and
streams the files, so the raw directory remains unchanged. “Immutable”
here is a process rule: the hash detects changed bytes but cannot prevent
a person or program from replacing them.


In [ ]:
local_integrity = {
    "archive_matches": (
        sha256_file(settings.archive_path) == settings.dataset.archive_sha256
    ),
    "csv_matches": (sha256_file(settings.csv_path) == settings.dataset.csv_sha256),
    "archive_sha256": settings.dataset.archive_sha256,
    "csv_sha256": settings.dataset.csv_sha256,
}
if not all(local_integrity[key] for key in ("archive_matches", "csv_matches")):
    raise RuntimeError(
        "Local data bytes differ from the reviewed course snapshot. "
        "Stop; do not explore, train, or reuse the old license review."
    )
local_integrity

## Inspect schema without exposing content

We need confirmed columns and file format, but not a screenful of raw
customer-like text. Reading only the header keeps this check bounded.


In [ ]:
import csv

with settings.csv_path.open(encoding="utf-8", newline="") as stream:
    columns = next(csv.reader(stream))
{"format": "CSV", "columns": columns}

## Exercise — make a scoped-use decision

Fill in the rationale. The safe default is deliberately narrow: local
learning is accepted under the recorded conditions; commercial,
enterprise, and production use remain unassessed.


In [ ]:
use_review = {
    "local_curriculum": "allowed_with_recorded_conditions",
    "commercial_use": "not_assessed",
    "production_suitability": "not_assessed",
    "redistribution": "retain_license_and_attribution_obligations",
    "rationale": (
        "Public availability and a recorded data license do not prove "
        "fitness, consent, accuracy, or policy acceptance for production."
    ),
}
assert use_review["production_suitability"] == "not_assessed"
use_review

**Hint:** separate permission to use data from evidence that the data is
accurate, representative, safe, and approved for a business purpose.


## Checkpoint

You have a reproducible source identity and a scoped use decision. A
future source version requires a fresh card and fresh hashes.

**Next:** `02_dataset_exploration_and_validation.ipynb` measures the
current source instead of trusting its description.
